In [1]:
!pip install ipywidgets

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, os
sys.path.insert(0, r"PROJECT_ROOT")

from ts_init import pro
import pandas as pd
import numpy as np
import joblib

# 直接定义函数（从 Day 3 复制，避免 joblib 路径问题）
def check_portfolio_health(portfolio_dict):
    df_mt = pd.read_csv("../data/600519_with_indicators.csv")
    df_mt["trade_date"] = pd.to_datetime(df_mt["trade_date"])
    df_mt = df_mt.sort_values("trade_date").reset_index(drop=True)
    
    report_lines = []
    for code, weight in portfolio_dict.items():
        if code != "600519.SH": continue
        latest = df_mt.iloc[-1]
        rsi = latest["rsi14"]
        rsi_status = "🔴 超买" if rsi > 70 else ("🟢 超卖" if rsi < 30 else "🟡 中性")
        ma_status = "🟢 多头" if latest["ma5"] > latest["ma20"] else "🔴 空头"
        weight_warning = "⚠️ 过重" if weight > 0.5 else "✅ 合理"
        report_lines.append(f"{code} | {weight*100:.0f}% | 价:{latest['close']:.0f} | RSI:{rsi:.1f}({rsi_status}) | {ma_status} | {weight_warning}")
    return "\n".join(report_lines)

print("✅ 持仓检查工具已就绪")

✅ Tushare 代理连接成功（30天试用账号）
✅ 持仓检查工具已就绪


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

mt_weight = widgets.FloatSlider(value=60, min=0, max=100, step=5, description='茅台权重(%)：')
hs300_weight = widgets.FloatSlider(value=40, min=0, max=100, step=5, description='沪深300权重(%)：')
btn = widgets.Button(description='🔍 开始体检', button_style='success')
output = widgets.Output()

def on_btn_click(b):
    with output:
        clear_output()
        portfolio = {
            "600519.SH": mt_weight.value / 100,
            "000300.SH": hs300_weight.value / 100
        }
        
        # 1. 原始报告
        report = check_portfolio_health(portfolio)
        print("📊 原始数据：")
        print(report)
        
        print("\n🤖 DeepSeek 投资顾问说：")
        # 2. 调用 DeepSeek 解读
        ai_talk = explain_report_with_ai(report)
        print(ai_talk)

btn.on_click(on_btn_click)

btn.on_click(on_btn_click)
display(widgets.VBox([mt_weight, hs300_weight, btn, output]))

In [1]:
!pip install openai

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from openai import OpenAI
import os

# 这里填你新生成的 DeepSeek Key
DEEPSEEK_KEY = "your_deepseek_api_key"  # ← 换成新 Key

client = OpenAI(
    api_key=DEEPSEEK_KEY,
    base_url="https://api.deepseek.com"
)

def explain_report_with_ai(report_text):
    """
    用 DeepSeek 解读持仓报告（中文效果很好）
    """
    response = client.chat.completions.create(
        model="deepseek-chat",  # 也可以用 deepseek-coder（代码更强）
        messages=[
            {
                "role": "system",
                "content": "你是资深A股投资顾问，用口语化、像朋友聊天一样的语言解读持仓体检报告，100字以内。"
            },
            {
                "role": "user",
                "content": report_text
            }
        ],
        temperature=0.7,
        max_tokens=200
    )
    return response.choices[0].message.content